In [1]:
# 04_ggi.ipynb
# Cross-Domain AI Governance Gap Analysis
# 04. Governance Gap Index (GGI) with noise correction
#
# For component k under regulation r with required threshold tau[r,k] and a score s_k
# oriented so higher is better:
#     g[r,k] = max(0, tau[r,k] - s_k) / tau[r,k]      (normalized shortfall in [0,1])
#     GGI[r] = sqrt( sum_k g[r,k]^2 )                 (lower is better; 0 = full compliance)
#
# Four component scores:
#   s_PERF : discriminative adequacy   = max(0, (AUC - 0.5) / 0.5)
#   s_DI   : fairness (worst-case DI)  = min(1, min over sensitive attrs of DisparateImpact)
#   s_SP   : fairness (worst-case SPD) = 1 - max over sensitive attrs of |StatParityDiff|
#   s_EXPL : explainability            = 1 - normalized entropy of the |SHAP| distribution
#
# NOISE CORRECTION (the primary GGI reported here). A near-chance model combined with many
# candidate proxies inflates the worst-case fairness axis: taking the minimum DI over many
# noisy, often tiny subgroups selects extreme values that pure noise reproduces (a floor /
# multiple-comparison artifact). We therefore gate the worst-case over sensitive attributes
# by significance: for each attribute a two-sided permutation test on the group difference
# in positive-prediction rate yields a p-value; Benjamini-Hochberg (FDR < 0.05) controls
# multiplicity within a dataset; the worst-case is taken ONLY over attributes that survive.
# If none survive there is no evidence of disparity and the axis scores 1.0 (parity).
# The performance axis is reported together with a label-permutation p-value for AUC > 0.5,
# so a weak-but-significant model is distinguished from one indistinguishable from chance.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
B_PERM = 2000             # permutations for the per-attribute fairness significance test
B_AUC = 500               # permutations for the AUC > 0.5 significance test
FDR_ALPHA = 0.05

# Regulatory strictness gradient (modeled scenario, not codified legal constants).
REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}
DATASETS = [
    ("German Credit",        "german"),
    ("Credit Card Approval", "credit_card_approval"),
    ("Medical Cost",         "insurance"),
    ("Porto Seguro",         "porto_seguro"),
]
REG_ORDER = ["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    return (pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet")),
            pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet")))

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    return X, df[target_col].astype(int)

def prevalence_threshold(proba, y_train):
    return np.quantile(proba, 1.0 - float(y_train.mean()))

def attr_rates(y_pred, priv_mask, unpriv_mask):
    pp = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    pu = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    return pp, pu

# Two-sided permutation p-value for an attribute's group difference in positive-prediction
# rate (statistic |ppr_unpriv - ppr_priv|). Shuffling predictions tests independence of
# prediction and group membership; this underlies both DI and SPD.
def perm_pvalue(y_pred, priv_mask, unpriv_mask, B=B_PERM, seed=RANDOM_STATE):
    pp, pu = attr_rates(y_pred, priv_mask, unpriv_mask)
    if not (np.isfinite(pp) and np.isfinite(pu)):
        return np.nan
    stat_obs = abs(pu - pp)
    r = np.random.default_rng(seed)
    n = len(y_pred)
    cnt = 0
    for _ in range(B):
        perm = y_pred[r.permutation(n)]
        p2 = perm[priv_mask].mean(); u2 = perm[unpriv_mask].mean()
        if abs(u2 - p2) >= stat_obs - 1e-12:
            cnt += 1
    return (cnt + 1) / (B + 1)

def auc_perm_pvalue(y_true, proba, B=B_AUC, seed=RANDOM_STATE):
    auc_obs = roc_auc_score(y_true, proba)
    r = np.random.default_rng(seed)
    yt = np.asarray(y_true)
    cnt = sum(roc_auc_score(r.permutation(yt), proba) >= auc_obs for _ in range(B))
    return auc_obs, (cnt + 1) / (B + 1)

def bh_reject(pvals, alpha=FDR_ALPHA):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    if n == 0:
        return np.array([], dtype=bool)
    order = np.argsort(p)
    passed = p[order] <= alpha * (np.arange(1, n + 1) / n)
    k = np.max(np.where(passed)[0]) if passed.any() else -1
    rej = np.zeros(n, dtype=bool)
    if k >= 0:
        rej[order[:k + 1]] = True
    return rej

# Per-attribute disparity table + raw / significance-gated worst-case fairness
def fairness_axes(test_df, sensitive_cols, y_pred):
    recs = []
    for col in sensitive_cols:
        s = test_df[col]
        if s.nunique(dropna=True) < 2:
            continue
        priv = s.mode(dropna=True).iloc[0]
        pm = (s == priv).values
        um = (s != priv).values & s.notna().values
        pp, pu = attr_rates(y_pred, pm, um)
        if not (np.isfinite(pp) and np.isfinite(pu)) or pp == 0:
            di = np.nan
        else:
            di = pu / pp
        spd = (pu - pp) if (np.isfinite(pp) and np.isfinite(pu)) else np.nan
        pval = perm_pvalue(y_pred, pm, um)
        recs.append({"attr": col, "DI": di, "absSPD": abs(spd) if np.isfinite(spd) else np.nan,
                     "p_perm": pval, "n_unpriv": int(um.sum())})
    rec_df = pd.DataFrame(recs)
    di_all = rec_df["DI"].dropna().values
    spd_all = rec_df["absSPD"].dropna().values
    sDI_raw = min(1.0, di_all.min()) if len(di_all) else np.nan
    sSP_raw = 1.0 - spd_all.max() if len(spd_all) else np.nan
    rej = bh_reject(np.where(np.isfinite(rec_df["p_perm"].values), rec_df["p_perm"].values, 1.0))
    rec_df = rec_df.assign(BH_significant=rej)
    sig = rec_df[rec_df["BH_significant"]]
    if len(sig) > 0:
        sDI_corr = min(1.0, np.nanmin(sig["DI"].values))
        sSP_corr = 1.0 - np.nanmax(sig["absSPD"].values)
    else:
        sDI_corr, sSP_corr = 1.0, 1.0
    return rec_df, sDI_raw, sSP_raw, sDI_corr, sSP_corr

def shap_explainability(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0 or len(mean_abs) <= 1:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    return float(1.0 - (-np.sum(p * np.log(p))) / np.log(len(mean_abs)))

def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

def ggi_value(sc, tau):
    gaps = [component_gap(sc[k], tau[k]) for k in ("s_PERF", "s_DI", "s_SP", "s_EXPL")]
    gaps = [g for g in gaps if np.isfinite(g)]
    return float(np.sqrt(np.sum(np.square(gaps)))) if gaps else np.nan

# Fit the unified baseline on each dataset; collect raw and corrected component scores.
comp_raw, comp_corr = {}, {}
sig_attrs = {}
porto_sig_table = None

for display_name, key in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    thr = prevalence_threshold(proba, y_train)
    y_pred = (proba >= thr).astype(int)

    auc, auc_p = auc_perm_pvalue(y_test.values, proba)
    s_PERF = max(0.0, (auc - 0.5) / 0.5)
    rec_df, sDI_raw, sSP_raw, sDI_corr, sSP_corr = fairness_axes(
        test, META[key]["sensitive_cols"], y_pred)
    s_EXPL = shap_explainability(model, X_test)
    domain = META[key].get("domain", "")

    comp_raw[display_name] = {"domain": domain, "AUC": auc, "AUC_perm_p": auc_p,
        "s_PERF": s_PERF, "s_DI": sDI_raw, "s_SP": sSP_raw, "s_EXPL": s_EXPL}
    comp_corr[display_name] = {"domain": domain, "AUC": auc, "AUC_perm_p": auc_p,
        "s_PERF": s_PERF, "s_DI": sDI_corr, "s_SP": sSP_corr, "s_EXPL": s_EXPL}
    sig_attrs[key] = rec_df.loc[rec_df["BH_significant"], "attr"].tolist()
    if key == "porto_seguro":
        porto_sig_table = rec_df.round(4)
    print(f"[{display_name:>22}] AUC={auc:.3f} (p={auc_p:.3f})  "
          f"s_DI raw={sDI_raw:.3f}->corr={sDI_corr:.3f}  "
          f"s_SP raw={sSP_raw:.3f}->corr={sSP_corr:.3f}  "
          f"n_sig={len(sig_attrs[key])}/{len(rec_df)}")

# Component-score tables (primary = noise-corrected; original kept for comparison)
def comp_table(store):
    return pd.DataFrame([
        {"domain": v["domain"], "dataset": k, "AUC": v["AUC"], "AUC_perm_p": v["AUC_perm_p"],
         "s_PERF": v["s_PERF"], "s_DI": v["s_DI"], "s_SP": v["s_SP"], "s_EXPL": v["s_EXPL"]}
        for k, v in store.items()]).round(4)

comp_corr_df = comp_table(comp_corr)
comp_raw_df = comp_table(comp_raw)
print(comp_corr_df.to_string(index=False))
comp_corr_df.to_csv(os.path.join(TAB_DIR, "04_component_scores.csv"), index=False)
comp_raw_df.to_csv(os.path.join(TAB_DIR, "04_component_scores_original.csv"), index=False)

# Per-(dataset x regulation) component gaps and GGI, for corrected and original scores
def ggi_full_table(store):
    rows = []
    for name, sc in store.items():
        for reg in REG_ORDER:
            tau = REGULATIONS[reg]
            g = {f"g_{k.split('_')[1]}": component_gap(sc[k], tau[k])
                 for k in ("s_PERF", "s_DI", "s_SP", "s_EXPL")}
            rows.append({"domain": sc["domain"], "dataset": name, "regulation": reg,
                         "g_PERF": g["g_PERF"], "g_DI": g["g_DI"], "g_SP": g["g_SP"],
                         "g_EXPL": g["g_EXPL"], "GGI": ggi_value(sc, tau)})
    return pd.DataFrame(rows).round(4)

ggi_corr_df = ggi_full_table(comp_corr)
ggi_raw_df = ggi_full_table(comp_raw)
print(ggi_corr_df.to_string(index=False))
ggi_corr_df.to_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"), index=False)
ggi_raw_df.to_csv(os.path.join(TAB_DIR, "04_ggi_full_original.csv"), index=False)

# Core matrix and domain summary (noise-corrected, primary)
ggi_matrix = (ggi_corr_df.pivot(index="dataset", columns="regulation", values="GGI")[REG_ORDER]).round(4)
print(ggi_matrix.to_string())
ggi_matrix.to_csv(os.path.join(TAB_DIR, "04_ggi_matrix.csv"))
domain_summary = (ggi_corr_df.groupby(["domain", "regulation"])["GGI"].mean().round(4)
                  .reset_index().pivot(index="domain", columns="regulation", values="GGI")[REG_ORDER])
domain_summary.to_csv(os.path.join(TAB_DIR, "04_ggi_by_domain.csv"))

# Corrected vs original domain means and insurance/credit ratio
cmp_rows = []
for reg in REG_ORDER:
    co = ggi_raw_df[ggi_raw_df.domain == "credit"].query("regulation==@reg")["GGI"].mean()
    io = ggi_raw_df[ggi_raw_df.domain == "insurance"].query("regulation==@reg")["GGI"].mean()
    cc = ggi_corr_df[ggi_corr_df.domain == "credit"].query("regulation==@reg")["GGI"].mean()
    ic = ggi_corr_df[ggi_corr_df.domain == "insurance"].query("regulation==@reg")["GGI"].mean()
    cmp_rows.append({"regulation": reg,
        "credit_original": round(co, 4), "insurance_original": round(io, 4), "ratio_original": round(io / co, 3),
        "credit_corrected": round(cc, 4), "insurance_corrected": round(ic, 4), "ratio_corrected": round(ic / cc, 3)})
cmp_df = pd.DataFrame(cmp_rows)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(os.path.join(TAB_DIR, "04_ggi_corrected_vs_original.csv"), index=False)

# Porto Seguro per-proxy disparity and BH significance (the floor / multiple-comparison check)
print(porto_sig_table.to_string(index=False))
porto_sig_table.to_csv(os.path.join(TAB_DIR, "04_porto_proxy_significance.csv"), index=False)

# Persist the BH-significant sensitive attributes per dataset for downstream notebooks
with open(os.path.join(TAB_DIR, "04_significant_attrs.json"), "w") as f:
    json.dump(sig_attrs, f, indent=2)
print("04_ggi complete")


[         German Credit] AUC=0.812 (p=0.002)  s_DI raw=1.000->corr=1.000  s_SP raw=0.907->corr=1.000  n_sig=0/2


[  Credit Card Approval] AUC=0.683 (p=0.002)  s_DI raw=1.000->corr=1.000  s_SP raw=0.968->corr=0.968  n_sig=1/1


[          Medical Cost] AUC=0.574 (p=0.024)  s_DI raw=0.661->corr=1.000  s_SP raw=0.899->corr=1.000  n_sig=0/3


[          Porto Seguro] AUC=0.568 (p=0.002)  s_DI raw=0.398->corr=0.398  s_SP raw=0.870->corr=0.966  n_sig=5/14
   domain              dataset    AUC  AUC_perm_p  s_PERF   s_DI   s_SP  s_EXPL
   credit        German Credit 0.8120       0.002  0.6241 1.0000 1.0000  0.1563
   credit Credit Card Approval 0.6835       0.002  0.3669 1.0000 0.9685  0.2162
insurance         Medical Cost 0.5742       0.024  0.1485 1.0000 1.0000  0.1374
insurance         Porto Seguro 0.5678       0.002  0.1355 0.3981 0.9659  0.0715
   domain              dataset      regulation  g_PERF   g_DI  g_SP  g_EXPL    GGI
   credit        German Credit KR_AI_Basic_Act  0.0000 0.0000   0.0  0.6874 0.6874
   credit        German Credit       EU_AI_Act  0.0000 0.0000   0.0  0.7395 0.7395
   credit        German Credit            NAIC  0.0000 0.0000   0.0  0.7158 0.7158
   credit Credit Card Approval KR_AI_Basic_Act  0.0000 0.0000   0.0  0.5676 0.5676
   credit Credit Card Approval       EU_AI_Act  0.0827 0.0000   0.0  0.6